In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from tensorflow import keras
from tensorflow.keras import layers
import uuid
import os

# Carregar o conjunto de dados
data = pd.read_csv('../data/diabetes_dataset.csv')

# Pré-processar variáveis categóricas
categorical_columns = ['gender', 'location', 'smoking_history']
data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)

# Separar características e alvo
X = data.drop('diabetes', axis=1)
y = data['diabetes']

# Normalizar características numéricas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir dados: 70% treino, 15% validação, 15% teste
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Definir uma função para criar, treinar e avaliar um modelo
def build_and_evaluate_model(model, model_name, X_train, y_train, X_val, y_val, X_test, y_test):
    # Compilar o modelo
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Parada antecipada
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    
    # Treinar o modelo
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=32, 
                        callbacks=[early_stopping], verbose=0)
    
    # Avaliar no conjunto de teste
    y_pred = (model.predict(X_test) > 0.5).astype(int)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    # Imprimir resultados
    print(f"\nResultados para {model_name}:")
    print(f"Acurácia: {accuracy:.4f}")
    print(f"Precisão: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    
    return model, accuracy, precision, recall

# Modelo 1: MLP simples (2 camadas ocultas, ReLU)
model1 = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Modelo 2: MLP mais profundo (3 camadas ocultas, ReLU)
model2 = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Modelo 3: MLP com tanh e dropout
model3 = keras.Sequential([
    layers.Dense(64, activation='tanh', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='tanh'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

# Treinar e avaliar todos os modelos
models = [(model1, "Modelo 1 (MLP Simples)"), (model2, "Modelo 2 (MLP Profundo)"), (model3, "Modelo 3 (Tanh + Dropout)")]
results = []

# Lista para armazenar métricas para o arquivo
metrics_output = []

for model, name in models:
    trained_model, accuracy, precision, recall = build_and_evaluate_model(model, name, X_train, y_train, X_val, y_val, X_test, y_test)
    results.append((trained_model, accuracy, name))
    # Adicionar métricas ao texto para o arquivo
    metrics_output.append(f"Resultados para {name}:\n")
    metrics_output.append(f"Acurácia: {accuracy:.4f}\n")
    metrics_output.append(f"Precisão: {precision:.4f}\n")
    metrics_output.append(f"Recall: {recall:.4f}\n\n")

# Selecionar o melhor modelo com base na acurácia
best_model, best_accuracy, best_model_name = max(results, key=lambda x: x[1])
print(f"\nMelhor modelo: {best_model_name} com acurácia {best_accuracy:.4f}")

# Adicionar informações do melhor modelo ao texto
metrics_output.append(f"Melhor modelo: {best_model_name} com acurácia {best_accuracy:.4f}\n")

# Salvar os pesos do melhor modelo
best_model.save_weights("../artifacts/best_model.weights.h5")
print(f"Pesos salvos para {best_model_name} como '../artifacts/best_model.weights.h5'")

# Salvar resultados em um arquivo resultados.txt
os.makedirs('../artifacts', exist_ok=True)  
with open('../artifacts/resultados.txt', 'w', encoding='utf-8') as f:
    f.writelines(metrics_output)
print("Resultados salvos em '../artifacts/resultados.txt'")

/home/codespace/.local/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 738us/step

Resultados para Modelo 1 (MLP Simples):
Acurácia: 0.9663
Precisão: 0.9190
Recall: 0.6495
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 713us/step

Resultados para Modelo 2 (MLP Profundo):
Acurácia: 0.9663
Precisão: 0.9268
Recall: 0.6430
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 691us/step

Resultados para Modelo 3 (Tanh + Dropout):
Acurácia: 0.9590
Precisão: 0.8744
Recall: 0.5890

Melhor modelo: Modelo 1 (MLP Simples) com acurácia 0.9663
Pesos salvos para Modelo 1 (MLP Simples) como '../artifacts/best_model.weights.h5'
Resultados salvos em '../artifacts/resultados.txt'
